In [ ]:
import os

import json
from pathlib import Path
import ijson
import pyarrow as pa
import pyarrow.parquet as pq

import pandas as pd
import numpy as np

import time
from datetime import datetime, timezone

import duckdb

In [ ]:
df_igdb = pd.read_json("igdb_games_raw.json") # df_games
df_games = pd.read_json("igdb_games_raw.json")

df_steam_map = pd.read_json("steam_map.json")

MANIFEST_PATH = Path("steam_etl") / "manifest.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(MANIFEST_PATH)

HISTORY_CSV_PATH = Path("itad_etl") / "itad_price_history.csv"
df_itad_history = pd.read_csv(HISTORY_CSV_PATH)

In [ ]:
PARQUET_DIR = Path("steam_etl") / "parquet_reviews"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 50_000

In [ ]:
def convert_json_to_parquet(path, batch_size=50_000):

    appid = int(path.stem.replace("appid_", ""))

    batch_records = []
    part = 0
    total_reviews = 0

    with path.open("rb") as f:

        for review in ijson.items(f, "reviews.item"):

            review["appid"] = appid
            batch_records.append(review)

            if len(batch_records) >= batch_size:

                df_batch = pd.json_normalize(batch_records)

                output_path = (
                    PARQUET_DIR /
                    f"appid_{appid}_part_{part:04d}.parquet"
                )

                df_batch.to_parquet(
                    output_path,
                    index=False,
                    compression="snappy"
                )

                total_reviews += len(df_batch)
                part += 1

                batch_records.clear()
                del df_batch

        # Guardar el último lote, aunque no llegue a 50.000
        if batch_records:

            df_batch = pd.json_normalize(batch_records)

            output_path = (
                PARQUET_DIR /
                f"appid_{appid}_part_{part:04d}.parquet"
            )

            df_batch.to_parquet(
                output_path,
                index=False,
                compression="snappy"
            )

            total_reviews += len(df_batch)

            del df_batch
            batch_records.clear()

    return {
        "appid": appid,
        "reviews_converted": total_reviews,
        "parquet_parts": len(
            list(PARQUET_DIR.glob(f"appid_{appid}_part_*.parquet"))
        )
    }

In [ ]:
REVIEWS_DIR = Path("steam_etl") / "reviews"
REVIEWS_DIR.mkdir(parents=True, exist_ok=True)

json_files = list(REVIEWS_DIR.glob("appid_*.json"))

largest_file = max(
    json_files,
    key=lambda p: p.stat().st_size
)

print("Archivo más grande:", largest_file.name)
print(
    "Tamaño (GB):",
    round(largest_file.stat().st_size / (1024 ** 3), 3)
)

In [ ]:
test_result = convert_json_to_parquet(
    largest_file,
    batch_size=BATCH_SIZE
)

print(test_result)

In [ ]:
game_type_names = {
    0: "main_game",             # Juego principal
    1: "dlc_addon",             # DLC o contenido adicional
    2: "expansion",             # Expansión
    3: "bundle",                # Paquete de juegos
    4: "standalone_expansion",  # Expansión independiente
    5: "mod",                   # Modificación
    6: "episode",               # Episodio
    7: "season",                # Temporada
    8: "remake",                # Remake
    9: "remaster",              # Remasterización
    10: "expanded_game",        # Juego ampliado
    11: "port",                 # Adaptación a otra plataforma
    12: "fork",                 # Derivado de otro juego
    13: "pack",                 # Pack
    14: "update"                # Actualización
}

df_games["game_type_name"] = df_games["game_type"].map(game_type_names)

df_games["game_type_name"].value_counts(dropna=False)

In [ ]:
# 1. Preparar los mapeos IGDB-Steam
mapping = df_steam_map.copy()

mapping["appid"] = pd.to_numeric(
    mapping["uid"], errors="coerce"
)

mapping = mapping.dropna(subset=["game", "appid"]).copy()
mapping["game"] = mapping["game"].astype(int)
mapping["appid"] = mapping["appid"].astype("int64")

mapping = (
    mapping.drop_duplicates(subset="game")
)

# 2. Unir la información de IGDB con el appid de Steam
candidates = df_games.copy()
candidates["id"] = candidates["id"].astype(int)

candidates = candidates.merge(
    mapping[["game", "appid"]].rename(columns={"game": "id"}),
    on="id",
    how="inner",
    validate="one_to_one"
)

candidates_main = candidates.loc[
    candidates["game_type_name"].eq("main_game")
].copy()

candidates_main = (
    candidates_main
    .drop_duplicates(subset="appid")
    .reset_index(drop=True)
)

print("Juegos candidatos:", len(candidates))
print("Main games:", len(candidates_main))
print("AppID únicos:", candidates_main["appid"].nunique())

In [ ]:
candidate_appids = set(
    pd.to_numeric(
        candidates_main["appid"],
        errors="coerce"
    )
    .dropna()
    .astype("int64")
    .unique()
)

In [ ]:
# Identificar los JSON correspondientes a los AppIDs en candidates_main

json_files = list(REVIEWS_DIR.glob("appid_*.json"))

json_files_main = [
    path
    for path in json_files
    if int(path.stem.replace("appid_", "")) in candidate_appids
]

json_files_outside = [
    path
    for path in json_files
    if int(path.stem.replace("appid_", "")) not in candidate_appids
]

print("JSON totales:", len(json_files))
print("JSON de candidates_main:", len(json_files_main))
print("JSON fuera de candidates_main:", len(json_files_outside))

In [ ]:
downloaded_appids = {
    int(path.stem.replace("appid_", ""))
    for path in json_files
}

candidate_appids_downloaded = candidate_appids.intersection(
    downloaded_appids
)

candidate_appids_pending = candidate_appids - downloaded_appids

print("Juegos en candidates_main:", len(candidate_appids))
print("Juegos con JSON descargado:", len(candidate_appids_downloaded))
print("Juegos pendientes:", len(candidate_appids_pending))

print(
    "Cobertura:",
    round(
        len(candidate_appids_downloaded) / len(candidate_appids) * 100,
        2
    ),
    "%"
)

In [ ]:
# Diccionario AppID -> ruta JSON
json_files_dict = {
    int(path.stem.replace("appid_", "")): path
    for path in json_files
}

# JSON que pertenecen a candidates_main
json_files_main = {
    appid: path
    for appid, path in json_files_dict.items()
    if appid in candidate_appids
}

In [ ]:
# Creamos un manifiesto paar controlar la conversión a Parquet

PARQUET_MANIFEST_PATH = PARQUET_DIR / "parquet_manifest.csv"

parquet_manifest_columns = [
    "appid",
    "status",
    "reviews_converted",
    "parquet_parts",
    "elapsed_seconds",
    "converted_at",
    "error"
]

if PARQUET_MANIFEST_PATH.exists():
    df_parquet_manifest = pd.read_csv(PARQUET_MANIFEST_PATH)

    # Asegurar que existen todas las columnas
    for col in parquet_manifest_columns:
        if col not in df_parquet_manifest.columns:
            df_parquet_manifest[col] = None

else:
    df_parquet_manifest = pd.DataFrame(
        columns=parquet_manifest_columns
    )

print("Registros en manifest:", len(df_parquet_manifest))

In [ ]:
# función para guardar el resultado de cada conevrsión
def update_parquet_manifest(result):

    global df_parquet_manifest

    appid = result["appid"]

    # Eliminar registro anterior del mismo AppID
    df_parquet_manifest = df_parquet_manifest.loc[
        df_parquet_manifest["appid"].astype(str) != str(appid)
    ].copy()

    # Añadir el resultado actualizado
    new_row = pd.DataFrame([result])

    df_parquet_manifest = pd.concat(
        [df_parquet_manifest, new_row],
        ignore_index=True
    )

    # Guardar manifest
    df_parquet_manifest.to_csv(
        PARQUET_MANIFEST_PATH,
        index=False
    )

In [ ]:
# AppIDs que ya constan como convertidos correctamente
completed_appids = set(
    pd.to_numeric(
        df_parquet_manifest.loc[
            df_parquet_manifest["status"].eq("success"),
            "appid"
        ],
        errors="coerce"
    )
    .dropna()
    .astype("int64")
)

# Contadores
converted = 0
skipped = 0
errors = 0
review_required = 0

# Resultados de la ejecución
conversion_results = []

# Recorrer los AppIDs ordenados
for i, (appid, path) in enumerate(
    sorted(json_files_main.items()),
    start=1
):

    print(f"\n[{i}/{len(json_files_main)}] AppID: {appid}")

    # 1. Comprobar si ya está convertido
    if appid in completed_appids:

        print("  -> Ya convertido. Se omite.")
        skipped += 1
        continue

    # 2. Comprobar si existen partes Parquet anteriores
    existing_parts = list(
        PARQUET_DIR.glob(f"appid_{appid}_part_*.parquet")
    )

    if existing_parts:

        print(
            f"  -> Hay {len(existing_parts)} partes Parquet existentes "
            "sin conversión registrada."
        )
        print("  -> Se omite para evitar duplicados. Revisar este AppID.")

        result = {
            "appid": appid,
            "status": "review_required",
            "reviews_converted": None,
            "parquet_parts": len(existing_parts),
            "elapsed_seconds": None,
            "converted_at": datetime.now().isoformat(),
            "error": "Existing Parquet files without successful manifest record"
        }

        update_parquet_manifest(result)

        review_required += 1
        continue

    # 3. Convertir JSON a Parquet
    start_time = time.time()

    try:

        result_conversion = convert_json_to_parquet(
            path,
            batch_size=BATCH_SIZE
        )

        elapsed = time.time() - start_time

        result = {
            "appid": appid,
            "status": "success",
            "reviews_converted": result_conversion["reviews_converted"],
            "parquet_parts": result_conversion["parquet_parts"],
            "elapsed_seconds": round(elapsed, 2),
            "converted_at": datetime.now().isoformat(),
            "error": None
        }

        converted += 1

        print(
            f"  -> Correcto: "
            f"{result['reviews_converted']:,} reviews "
            f"en {result['parquet_parts']} partes "
            f"({elapsed:.2f} segundos)"
        )

    except Exception as e:

        elapsed = time.time() - start_time

        result = {
            "appid": appid,
            "status": "error",
            "reviews_converted": None,
            "parquet_parts": None,
            "elapsed_seconds": round(elapsed, 2),
            "converted_at": datetime.now().isoformat(),
            "error": str(e)
        }

        errors += 1

        print(f"  -> ERROR: {e}")

    # 4. Guardar resultado inmediatamente
    update_parquet_manifest(result)

    conversion_results.append(result)


print("\n" + "=" * 50)
print("RESUMEN DE LA CONVERSIÓN")
print("=" * 50)

print("Convertidos:", converted)
print("Ya convertidos (omitidos):", skipped)
print("Con errores:", errors)
print("Pendientes de revisión:", review_required)

print("Manifest:", PARQUET_MANIFEST_PATH)

In [ ]:
# Comprobamos los resultados
df_parquet_manifest = pd.read_csv(PARQUET_MANIFEST_PATH)

print("Estados de conversión:")
print(df_parquet_manifest["status"].value_counts(dropna=False))

print("\nTotal de juegos convertidos:")
print(
    df_parquet_manifest.loc[
        df_parquet_manifest["status"].eq("success"),
        "appid"
    ].nunique()
)

print("\nTotal de reviews convertidas:")
print(
    df_parquet_manifest.loc[
        df_parquet_manifest["status"].eq("success"),
        "reviews_converted"
    ].sum()
)

print("\nTotal de partes Parquet:")
print(
    df_parquet_manifest.loc[
        df_parquet_manifest["status"].eq("success"),
        "parquet_parts"
    ].sum()
)

In [ ]:
# Comprobamos qué AppID han dado errores
df_parquet_errors = df_parquet_manifest.loc[
    df_parquet_manifest["status"].isin(["error", "review_required"])
]

df_parquet_errors[
    ["appid", "status", "error"]
].head(20)

In [ ]:
REVIEWS_DIR = Path("steam_etl/reviews")
EDA_DIR = Path("eda_data")
EDA_DIR.mkdir(parents=True, exist_ok=True)

def extract_steam_summary(json_path):

    fields = {
        "appid": None,
        "extracted_at": None,
        "total_positive": None,
        "total_negative": None,
        "total_reviews": None,
        "review_count_downloaded": None
    }

    field_mapping = {
        "appid": "appid",
        "extracted_at": "extracted_at",
        "query_summary.total_positive": "total_positive",
        "query_summary.total_negative": "total_negative",
        "query_summary.total_reviews": "total_reviews",
        "review_count_downloaded": "review_count_downloaded"
    }

    with open(json_path, "rb") as f:

        for prefix, event, value in ijson.parse(f):

            # Detener la lectura antes de las reseñas
            if prefix == "reviews" and event == "start_array":
                break

            if prefix in field_mapping and event in (
                "number", "string", "null"
            ):
                fields[field_mapping[prefix]] = value

    if fields["appid"] is None:
        fields["appid"] = int(
            json_path.stem.replace("appid_", "")
        )

    return fields

In [ ]:
# Extraemos los resúmenes de los JSON de Steam
json_files = sorted(
    REVIEWS_DIR.glob("appid_*.json")
)

summaries = []
errors = []

for i, path in enumerate(json_files, start=1):

    try:
        summaries.append(
            extract_steam_summary(path)
        )

    except Exception as e:
        errors.append({
            "file": path.name,
            "error": str(e)
        })

    if i % 1000 == 0:
        print(f"Procesados: {i:,}/{len(json_files):,}")

df_steam_summary = pd.DataFrame(summaries)

numeric_cols = [
    "appid",
    "total_positive",
    "total_negative",
    "total_reviews",
    "review_count_downloaded"
]

for col in numeric_cols:
    df_steam_summary[col] = pd.to_numeric(
        df_steam_summary[col],
        errors="coerce"
    )

print("Videojuegos procesados:", len(df_steam_summary))
print("Errores:", len(errors))

display(df_steam_summary.head())

In [ ]:
df_steam_summary.to_parquet(
    EDA_DIR / "steam_reviews_summary.parquet",
    index=False
)

####################################################

In [ ]:
# Probamos la integración de parquet y duckdb

con = duckdb.connect("steam_analytics.duckdb")

df_review_stats = con.execute("""
    SELECT
        COUNT(DISTINCT appid) AS total_games,
        COUNT(*) AS total_reviews,
        AVG(CAST(voted_up AS INTEGER)) AS positive_review_ratio
    FROM read_parquet(
        'steam_etl/parquet_reviews/*.parquet',
        union_by_name = true
    )
""").df()

display(df_review_stats)

In [ ]:
df_reviews = con.execute("""
    SELECT
        appid,
        COUNT(*) AS total_reviews
    FROM read_parquet(
        'steam_etl/parquet_reviews/appid_*.parquet'
    )
    GROUP BY appid
""").df()

display(df_reviews)

####################################################

In [ ]:
ITAD_DIR = Path("itad_etl")
HISTORY_DIR = ITAD_DIR / "history"
PARQUET_DIR = ITAD_DIR / "parquet_history"

PARQUET_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_MANIFEST_PATH = PARQUET_DIR / "parquet_manifest.csv"

BATCH_SIZE = 50_000

MANIFEST_COLUMNS = [
    "appid",
    "status",
    "records_converted",
    "parquet_parts",
    "elapsed_seconds",
    "converted_at",
    "error"
]

In [ ]:
json_files_itad = {
    int(path.stem.replace("history_", "")): path
    for path in HISTORY_DIR.glob("history_*.json")
}

print("JSON de ITAD encontrados:", len(json_files_itad))

# Inspeccionar el primer JSON con registros
for appid, path in sorted(json_files_itad.items()):

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("AppID:", appid)
    print("Tipo de raíz:", type(data).__name__)

    if isinstance(data, dict):
        print("Claves principales:", list(data.keys()))

        for key, value in data.items():
            if isinstance(value, list):
                print(f"Lista '{key}': {len(value)} elementos")
                if value:
                    print("Primer registro:", value[0])
                    break

    elif isinstance(data, list):
        print("Registros:", len(data))
        if data:
            print("Primer registro:", data[0])

    break

In [ ]:
if PARQUET_MANIFEST_PATH.exists():

    df_parquet_manifest = pd.read_csv(
        PARQUET_MANIFEST_PATH
    )

else:

    df_parquet_manifest = pd.DataFrame(
        columns=MANIFEST_COLUMNS
    )

In [ ]:
def update_parquet_manifest(result, max_retries=5):

    global df_parquet_manifest

    appid = int(result["appid"])

    # Eliminar el registro anterior de este AppID
    df_updated = df_parquet_manifest.loc[
        pd.to_numeric(
            df_parquet_manifest["appid"],
            errors="coerce"
        ).ne(appid)
    ].copy()

    # Añadir el resultado nuevo
    df_updated = pd.concat(
        [
            df_updated,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    temp_path = PARQUET_MANIFEST_PATH.with_suffix(".csv.tmp")

    # Escribir manifest temporal
    df_updated.to_csv(
        temp_path,
        index=False
    )

    # Intentar reemplazar el archivo
    for attempt in range(max_retries):

        try:
            os.replace(
                temp_path,
                PARQUET_MANIFEST_PATH
            )

            df_parquet_manifest = df_updated

            return

        except PermissionError:

            if attempt == max_retries - 1:
                raise

            wait = 2 * (attempt + 1)

            print(
                f"Manifest bloqueado. "
                f"Reintento {attempt + 1}/{max_retries} "
                f"en {wait}s..."
            )

            time.sleep(wait)

In [ ]:
def convert_itad_json_to_parquet(
    json_path,
    batch_size=50_000
):

    json_path = Path(json_path)

    appid = int(
        json_path.stem.replace("history_", "")
    )

    # 1. Leer el JSON de este videojuego
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 2. Identificar los registros históricos
    if isinstance(data, list):

        records = data
        itad_id = None

    elif isinstance(data, dict):

        itad_id = data.get("itad_id")

        records = None

        for key in ["records", "history", "data", "prices"]:

            if isinstance(data.get(key), list):
                records = data[key]
                break

        if records is None:
            raise ValueError(
                f"Estructura JSON no reconocida: {list(data.keys())}"
            )

    else:
        raise ValueError("Formato JSON no válido")

    total_records = len(records)

    # 3. Si no tiene historial, no crear Parquet vacío
    if total_records == 0:

        return {
            "records_converted": 0,
            "parquet_parts": 0
        }

    # 4. Convertir en bloques
    parquet_parts = 0
    records_converted = 0
    temp_files = []

    try:

        for start in range(0, total_records, batch_size):

            batch = records[start:start + batch_size]

            df_batch = pd.json_normalize(batch, sep="_")

            # Identificadores para posteriores JOIN
            df_batch["appid"] = appid

            if itad_id is not None:
                df_batch["itad_id"] = str(itad_id)

            table = pa.Table.from_pandas(
                df_batch,
                preserve_index=False
            )

            output_path = (
                PARQUET_DIR
                / f"appid_{appid}_part_{parquet_parts:04d}.parquet"
            )

            temp_path = output_path.with_suffix(".parquet.tmp")

            pq.write_table(
                table,
                temp_path,
                compression="snappy"
            )

            temp_files.append((temp_path, output_path))

            records_converted += len(df_batch)
            parquet_parts += 1

        # 5. Publicar los archivos una vez escritos los lotes
        for temp_path, output_path in temp_files:
            os.replace(temp_path, output_path)

    except Exception:

        # Eliminar solo archivos temporales de esta ejecución
        for temp_path, _ in temp_files:
            temp_path.unlink(missing_ok=True)

        raise

    return {
        "records_converted": records_converted,
        "parquet_parts": parquet_parts
    }

In [ ]:
completed_appids = set(
    pd.to_numeric(
        df_parquet_manifest.loc[
            df_parquet_manifest["status"].eq("success"),
            "appid"
        ],
        errors="coerce"
    )
    .dropna()
    .astype("int64")
)

converted = 0
skipped = 0
errors = 0
review_required = 0

conversion_results = []

for i, (appid, path) in enumerate(
    sorted(json_files_itad.items()),
    start=1
):

    print(f"\n[{i}/{len(json_files_itad)}] AppID: {appid}")

    # 1. Omitir conversiones completadas
    if appid in completed_appids:

        print("  -> Ya convertido. Se omite.")
        skipped += 1
        continue

    # 2. Comprobar si existen partes anteriores
    existing_parts = list(
        PARQUET_DIR.glob(
            f"appid_{appid}_part_*.parquet"
        )
    )

    if existing_parts:

        print("  -> Existen Parquet sin conversión confirmada.")

        result = {
            "appid": appid,
            "status": "review_required",
            "records_converted": None,
            "parquet_parts": len(existing_parts),
            "elapsed_seconds": None,
            "converted_at": datetime.now().isoformat(),
            "error": "Existing Parquet files without successful manifest"
        }

        review_required += 1

        update_parquet_manifest(result)
        conversion_results.append(result)

        continue

    # 3. Convertir JSON a Parquet
    start_time = time.time()

    try:

        result_conversion = convert_itad_json_to_parquet(
            path,
            batch_size=BATCH_SIZE
        )

        elapsed = time.time() - start_time

        result = {
            "appid": appid,
            "status": "success",
            "records_converted": result_conversion["records_converted"],
            "parquet_parts": result_conversion["parquet_parts"],
            "elapsed_seconds": round(elapsed, 2),
            "converted_at": datetime.now().isoformat(),
            "error": None
        }

        converted += 1

        print(
            f"  -> Correcto: "
            f"{result['records_converted']:,} registros "
            f"en {result['parquet_parts']} partes "
            f"({elapsed:.2f} segundos)"
        )

    except Exception as e:

        elapsed = time.time() - start_time

        result = {
            "appid": appid,
            "status": "error",
            "records_converted": None,
            "parquet_parts": None,
            "elapsed_seconds": round(elapsed, 2),
            "converted_at": datetime.now().isoformat(),
            "error": str(e)
        }

        errors += 1

        print(f"  -> ERROR: {e}")

    # 4. Actualizar manifiesto
    update_parquet_manifest(result)

    conversion_results.append(result)


print("\n" + "=" * 50)
print("RESUMEN DE LA CONVERSIÓN ITAD")
print("=" * 50)

print("Convertidos:", converted)
print("Ya convertidos:", skipped)
print("Errores:", errors)
print("Pendientes de revisión:", review_required)
print("Manifest:", PARQUET_MANIFEST_PATH)

In [ ]:
df_parquet_manifest = pd.read_csv(
    PARQUET_MANIFEST_PATH
)

print("--- ESTADOS ---")
display(
    df_parquet_manifest["status"].value_counts()
)

print("\n--- REGISTROS CONVERTIDOS ---")

print(
    "Total registros:",
    int(
        df_parquet_manifest.loc[
            df_parquet_manifest["status"] == "success",
            "records_converted"
        ].sum()
    )
)

print(
    "Total partes Parquet:",
    len(list(PARQUET_DIR.glob("*.parquet")))
)

print(
    "AppIDs sin registros históricos:",
    (
        (df_parquet_manifest["status"] == "success")
        & (df_parquet_manifest["records_converted"] == 0)
    ).sum()
)

####################################################

In [ ]:
df_preview = con.execute("""
    SELECT *
    FROM read_parquet(
        'itad_etl/parquet_history/*.parquet',
        union_by_name = true
    )
    LIMIT 5
""").df()

display(df_preview)

####################################################